In [16]:
import requests
from bs4 import BeautifulSoup, Comment
import pandas as pd
import os
class RosterAndSplits:
    def __init__(self, team, year, output_folder="output"):
        self.team = team
        self.year = year
        self.output_folder = output_folder
        self.urls = []
        self.all_rows = dict()
        self.all_cols = dict()
    def generate_urls(self):
        self.urls.append("https://www.pro-football-reference.com/teams/" + self.team + "/" + str(self.year) + "_roster.htm")
        self.urls.append("https://www.pro-football-reference.com/teams/" + self.team + "/" + str(self.year) + "_travel.htm")
        self.urls.append("https://www.pro-football-reference.com/teams/" + self.team + "/" + str(self.year) + "_splits.htm")
        self.urls.append("https://www.pro-football-reference.com/teams/" + self.team + "/" + str(self.year) + "_opp_splits.htm")
    @staticmethod
    def all_tables(url):
        response = requests.get(url)
        text = BeautifulSoup(response.content, 'html.parser')
        table_ids = [table.get("id") for table in text.find_all("table") if table.get("id")]
        comments = text.find_all(string=lambda text: isinstance(text, Comment))
        hidden_tables = []
        for c in comments:
            comment_soup = BeautifulSoup(c, "html.parser")
            tables = comment_soup.find_all("table")
            for table in tables:
                tid = table.get("id")
                if tid:
                    hidden_tables.append(table)
        return table_ids, hidden_tables, text
    @staticmethod
    def find_duplicates(input_list):
        seen = set()
        duplicates = dict()
        for i in range(len(input_list)):
            item = input_list[i]
            if item not in seen:
                seen.add(item)
            else:
                if item in duplicates:
                    duplicates[item][0] += 1
                    duplicates[item][1].append(i)
                else:
                    duplicates[item] = [1,[i]]
        return duplicates
    @staticmethod
    def rows_and_columns(all_rows, all_cols, text, table_ids, hidden_tables):
        for id in table_ids:

            cur = text.find('table', {'id': id})
            cur_rows= cur.find_all('tr')
            if id in all_rows:
                all_rows[id].extend(cur_rows)
            else:
                all_rows[id] = cur_rows
            columnT = cur.find_all('th')
            cols = []
            for title in columnT:
                header = title.get('aria-label')
                cols.append(header)
                all_cols[id] = cols
        if(len(hidden_tables) > 0):
            for table in hidden_tables:
                id = table.get("id")
                cur_rows= table.find_all('tr')
                if id in all_rows:
                    all_rows[id].extend(cur_rows)
                else:
                    all_rows[id] = cur_rows
                columnT = table.find_all('th')
                cols = []
                for title in columnT:
                    header = title.get('aria-label')
                    cols.append(header)
                    all_cols[id] = cols

    def evaluate(self):
        self.generate_urls()
        for url in self.urls:
            table_ids, hidden_tables, text = self.all_tables(url)
            self.rows_and_columns(self.all_rows, self.all_cols, text, table_ids, hidden_tables)
        for id in self.all_rows:
            flag = ["starters", "rosters", "map_table", "vegas_lines"]
            bool = False
            if id in flag:
                bool = True
            headers = []
            act = []
            data = []
            rows = self.all_rows[id]
            cols = self.all_cols[id]
            for row in rows:
                cols = row.find_all(['td', 'th'])
                cols = [col.text.strip() for col in cols]
                data.append(cols)
            if (bool):
                headers = data[0]
            else:
                headers = data[1]
            duplicates =  self.find_duplicates(headers)
            if (len(duplicates) > 0):
                for item in duplicates:
                    count = duplicates[item][0]
                    all_duplicates = duplicates[item][1]
                    cur_word = item
                    for i in range(count):
                        idx = all_duplicates[i]
                        cur_word += 'R'
                        headers[idx] = cur_word
            df = pd.DataFrame(data, columns=[headers])  # Replace with actual column names
            if bool:
                df = df.drop([0])
            else:
                df = df.drop([0,1])
            df = df.reset_index(drop=True)
            folder_path = self.output_folder
            file_name = id + '1.csv'
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)
            path = os.path.join(folder_path, file_name)
            df.to_csv(path, index=False)





    

In [10]:
#test
roster_2004 = RosterAndSplits("clt", "2004", "Roster1")

In [11]:
roster_2004.evaluate()

In [17]:
#test1
roster_2024 = RosterAndSplits("clt", "2024", "Roster2")

In [18]:
roster_2024.evaluate()